# R matrix reconstruction from beam-size data

This notebook follows the equation from the multiple-wire / multi-screen emittance measurement method.
The same equation can be used for a quadrupole scan.

For one transverse plane, the beam matrix at the starting point `s0` is

$$
\Sigma_0 =
\epsilon
\begin{pmatrix}
\beta_0 & -\alpha_0 \\
-\alpha_0 & \gamma_0
\end{pmatrix},
\qquad
\gamma_0 = \frac{1 + \alpha_0^2}{\beta_0}
$$

The linear transport from `s0` to screen `i` is

$$
R^{(i)} =
\begin{pmatrix}
R_{11}^{(i)} & R_{12}^{(i)} \\
R_{21}^{(i)} & R_{22}^{(i)}
\end{pmatrix}.
$$

The beam matrix at screen `i` is transported as

$$
\Sigma_i = R^{(i)} \Sigma_0 \left(R^{(i)}\right)^T.
$$

The measured beam size squared is the first element of this matrix:

$$
\sigma_i^2 = \Sigma_{i,11}.
$$

After expanding this, we get

$$
\sigma_i^2 =
\left(R_{11}^{(i)}\right)^2 \beta_0 \epsilon
+ 2R_{11}^{(i)}R_{12}^{(i)}\left(-\alpha_0\epsilon\right)
+ \left(R_{12}^{(i)}\right)^2 \gamma_0\epsilon.
$$

So for many screens or many measurements, this becomes a linear system:

$$
\begin{pmatrix}
\sigma_1^2 \\
\sigma_2^2 \\
\sigma_3^2 \\
\vdots \\
\sigma_n^2
\end{pmatrix}
=
\begin{pmatrix}
\left(R_{11}^{(1)}\right)^2 & 2R_{11}^{(1)}R_{12}^{(1)} & \left(R_{12}^{(1)}\right)^2 \\
\left(R_{11}^{(2)}\right)^2 & 2R_{11}^{(2)}R_{12}^{(2)} & \left(R_{12}^{(2)}\right)^2 \\
\left(R_{11}^{(3)}\right)^2 & 2R_{11}^{(3)}R_{12}^{(3)} & \left(R_{12}^{(3)}\right)^2 \\
\vdots & \vdots & \vdots \\
\left(R_{11}^{(n)}\right)^2 & 2R_{11}^{(n)}R_{12}^{(n)} & \left(R_{12}^{(n)}\right)^2
\end{pmatrix}
\begin{pmatrix}
\beta_0\epsilon \\
-\alpha_0\epsilon \\
\gamma_0\epsilon
\end{pmatrix}.
$$

In our dataset, we do the inverse problem: we know many input beam parameters and many output beam sizes, so we fit the coefficients multiplying

$$
\beta\epsilon, \qquad -\alpha\epsilon, \qquad \gamma\epsilon.
$$

Therefore, for each screen we fit

$$
\sigma_i^2 = A_i\beta\epsilon + B_i(-\alpha\epsilon) + C_i\gamma\epsilon
$$

For ideal uncoupled linear transport:

$$
A_i = \left(R_{11}^{(i)}\right)^2,
\qquad
B_i = 2R_{11}^{(i)}R_{12}^{(i)},
\qquad
C_i = \left(R_{12}^{(i)}\right)^2.
$$

For the horizontal plane we solve

$$
B_x = C_x R_x^T,
$$

where

$$
C_x =
\begin{pmatrix}
\beta_x\epsilon_x & -\alpha_x\epsilon_x & \gamma_x\epsilon_x & 1
\end{pmatrix}
$$

and

$$
B_x =
\begin{pmatrix}
\sigma_{x,OTR0}^2 & \sigma_{x,OTR1}^2 & \sigma_{x,OTR2}^2 & \sigma_{x,OTR3}^2
\end{pmatrix}.
$$

Similarly for the vertical plane:

$$
B_y = C_y R_y^T.
$$


In [ ]:
import numpy as np
import RF_Track as rft

dataset = np.load("../../MachineLearning/R_matrix_dataset_fixedK1.npz")
I = dataset["X"]
O = dataset["Y"]

mass = rft.electronmass  # MeV/c^2
momentum = 1300  # MeV/c

beta_gamma = momentum / mass

emitt_x = I[:, 0] / beta_gamma  # mm.mrad, geometric emittance
emitt_y = I[:, 3] / beta_gamma  # mm.mrad, geometric emittance
beta_x = I[:, 1]
beta_y = I[:, 4]
alpha_x = I[:, 2]
alpha_y = I[:, 5]

S1_x = O[:, 0]  # mm
S1_y = O[:, 1]
S2_x = O[:, 2]
S2_y = O[:, 3]
S3_x = O[:, 4]
S3_y = O[:, 5]
S4_x = O[:, 6]
S4_y = O[:, 7]

gamma_x = (1 + alpha_x**2) / beta_x
gamma_y = (1 + alpha_y**2) / beta_y

ones = np.ones(len(I))

Cx = np.column_stack((beta_x * emitt_x, -alpha_x * emitt_x, gamma_x * emitt_x, ones))
Cy = np.column_stack((beta_y * emitt_y, -alpha_y * emitt_y, gamma_y * emitt_y, ones))

Bx = np.column_stack((S1_x, S2_x, S3_x, S4_x)) ** 2
By = np.column_stack((S1_y, S2_y, S3_y, S4_y)) ** 2

Rx = Bx.T @ np.linalg.pinv(Cx.T)
Ry = By.T @ np.linalg.pinv(Cy.T)

print("Rx =")
print(Rx)

print("Ry =")
print(Ry)

In [ ]:
coeffs_x = np.linalg.lstsq(Cx, Bx, rcond = None)[0]
coeffs_y = np.linalg.lstsq(Cy, By, rcond = None)[0]

Rx_fit = coeffs_x.T
Ry_fit = coeffs_y.T

print("Rx_fit:")
print("columns: R11^2, 2R11R12, R12^2, offset")
print(Rx_fit)

print("Ry_fit:")
print("columns: R33^2, 2R33R34, R34^2, offset")
print(Ry_fit)

In [ ]:
Bx_predict = Cx @ coeffs_x
By_predict = Cy @ coeffs_y

res_x = Bx_predict - Bx
res_y = By_predict - By

print("relative RMS x:", np.sqrt(np.mean(res_x**2, axis=0)) / np.mean(Bx, axis=0))
print("relative RMS y:", np.sqrt(np.mean(res_y**2, axis=0)) / np.mean(By, axis=0))

Errors are of magnitude 0.1% - 0.4%, which means we can treat the problem as linear (e.g. no sextupoles between the screens) and that:

$$
\sigma_i^2 = A_i\beta\epsilon + B_i(-\alpha\epsilon) + C_i\gamma\epsilon
$$

is really explaining the system.

In [ ]:
twiss_path = "../../Interfaces/ATF2/Ext_ATF2/ATF2_EXT_FF_v5.2.twiss"
quad_name = "QD18X"
screens = list(dataset["screens"])

with open(twiss_path, "r") as f:
    lines = [line.strip() for line in f if line.strip()]

header_i = next(i for i, line in enumerate(lines) if line.startswith("*"))
format_i = next(i for i, line in enumerate(lines) if line.startswith("$") and i > header_i)
columns = lines[header_i].lstrip("*").split()

idx_name = columns.index("NAME")
idx_betx = columns.index("BETX")
idx_alfx = columns.index("ALFX")
idx_mux = columns.index("MUX")
idx_bety = columns.index("BETY")
idx_alfy = columns.index("ALFY")
idx_muy = columns.index("MUY")

T = {}
for line in lines[format_i + 1:]:
    if line.startswith(("@", "*", "$")):
        continue
    data = line.split()
    name = data[idx_name].strip('"')
    row = {
        "betx": float(data[idx_betx]),
        "alfx": float(data[idx_alfx]),
        "mux": float(data[idx_mux]),
        "bety": float(data[idx_bety]),
        "alfy": float(data[idx_alfy]),
        "muy": float(data[idx_muy]),
    }
    T.setdefault(name, []).append(row)


def R_from_twiss(start, end, plane):
    start_row = T[start][0]
    end_row = T[end][-1]

    if plane == "x":
        beta0, alpha0, mu0 = start_row["betx"], start_row["alfx"], start_row["mux"]
        beta1, mu1 = end_row["betx"], end_row["mux"]
    else:
        beta0, alpha0, mu0 = start_row["bety"], start_row["alfy"], start_row["muy"]
        beta1, mu1 = end_row["bety"], end_row["muy"]

    dmu = 2 * np.pi * (mu1 - mu0)
    R11 = np.sqrt(beta1 / beta0) * (np.cos(dmu) + alpha0 * np.sin(dmu))
    R12 = np.sqrt(beta0 * beta1) * np.sin(dmu)
    return R11, R12

for screen in screens:
    print(screen, "occurrences in Twiss:", len(T.get(screen, [])))

R_model_x = np.array([R_from_twiss(quad_name, screen, "x") for screen in screens])
R_model_y = np.array([R_from_twiss(quad_name, screen, "y") for screen in screens])

print("RF-Track/Twiss model R_x = [R11, R12]")
for screen, row in zip(screens, R_model_x):
    print(screen, row)

print("\nRF-Track/Twiss model R_y = [R33, R34]")
for screen, row in zip(screens, R_model_y):
    print(screen, row)

Rx_model_coeffs = np.column_stack((
    R_model_x[:, 0] ** 2,
    2 * R_model_x[:, 0] * R_model_x[:, 1],
    R_model_x[:, 1] ** 2,
))

Ry_model_coeffs = np.column_stack((
    R_model_y[:, 0] ** 2,
    2 * R_model_y[:, 0] * R_model_y[:, 1],
    R_model_y[:, 1] ** 2,
))

print("Horizontal comparison")
print("screen | fit [R11^2, 2R11R12, R12^2] | model | diff")
for screen, fit, model in zip(screens, Rx[:, :3], Rx_model_coeffs):
    print(screen, fit, model, fit - model)

print("\nVertical comparison")
print("screen | fit [R33^2, 2R33R34, R34^2] | model | diff")
for screen, fit, model in zip(screens, Ry[:, :3], Ry_model_coeffs):
    print(screen, fit, model, fit - model)